<a href="https://colab.research.google.com/github/LayanYasiru/LangChain/blob/main/A_I_Bot_With_History.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

We use langchain -qU for: installing the main framework to build chains, agents, and manage your application's logic.

We use langchain-google-genai -qU for: installing the specific connectors needed to let that framework talk to Google's Gemini models.

In [27]:
!pip install langchain -qU
!pip install langchain-google-genai -qU



**1. import os**

    Used for: interacting with the operating system. In this context, it is often used to set environment variables (like os.environ), though the next line suggests you are using Colab's built-in secrets manager instead.

**2. from google.colab import userdata**

    Used for: securely accessing your API keys stored in the "Secrets" section of Google Colab, keeping them hidden from the code.

**3. from langchain_google_genai import ChatGoogleGenerativeAI**

    Used for: importing the specific class that lets you chat with Google's Gemini models. This is the connector that sends your prompt to Google and gets the answer back.

**4. from langchain_core.prompts import ChatPromptTemplate**

    Used for: creating structured "templates" for your prompts. It allows you to define a standard format (like "Translate {text} to French") and dynamically swap in different variables later.

**5. from langchain_core.output_parsers import StrOutputParser**

    Used for: taking the complex response object from the AI (which contains metadata, finish reasons, etc.) and extracting just the final string (the actual text answer) so it is easier to read.

In [28]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [29]:
os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')


**ChatGoogleGenerativeAI(...)**

What it does: Calls the class you imported earlier to build the actual connection to Google's servers.

**model="gemini-2.5-flash"**

What it does: Tells Google exactly which version of the "brain" you want to use.

Note: "Flash" models are typically designed to be faster and more cost-effective compared to "Pro" models.

**temperature=0**

What it does: Controls how "creative" or "random" the AI is.

Value 0: This makes the model deterministic. It will try to give the most factual, precise, and consistent answer possible. It is best for coding, math, or data extraction.

Higher Values (e.g., 0.7): make the model more creative and varied, better for writing stories or brainstorming.

In [30]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


```python
prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are an intelligent chatbot. Answer the following question."),
        ("user", "{question}")
    ]
)
```



---


**`ChatPromptTemplate.from_messages(...)`**

**What it does:**
Creates a structured prompt template that defines how messages are sent to the AI model. It helps control the conversation format between system and user.

---

**`("system", "You are an intelligent chatbot. Answer the following question.")`**

**What it does:**
Defines the **system message**, which sets the **behavior and personality** of the AI.

* It tells the AI:

  * It is an intelligent chatbot
  * It must focus on answering questions clearly
* This message is **always considered first** by the model.
* It is used to control:

  * Tone
  * Rules
  * Expertise
  * Behavior

---

**`("user", "{question}")`**

**What it does:**
Defines the **user message template**.

* `{question}` is a **dynamic variable**
* The actual question will be injected at runtime like this:

  ```python
  prompt.format(question="What is Machine Learning?")
  ```
* This makes your prompt **reusable for any question**


In [31]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are an intelligent chatbot. Answer the following question."),
        ("user", "{question}")
    ]
)

**StrOutputParser()**

What it does: Initializes the String Output Parser tool.

Why use it:

Without this: The AI returns a complex object called an AIMessage that looks like this: content='Hello!' response_metadata={'token_usage': ...} id='run-123'

With this: It automatically extracts just the content part, so you get a clean, simple string like: 'Hello!'

In [32]:
parser = StrOutputParser()


# The Flow of Data

Step 1 (prompt): The chain starts here. It receives your input (e.g., {"question": "What is AI?"}), fills in the template, and creates a list of formatted messages.

Step 2 (llm): The formatted messages are piped into the LLM. The model thinks and outputs a complex AIMessage object.

Step 3 (parser): That complex object is piped into the Parser. The parser strips away the metadata and keeps only the final text string.

chain =

What it does: It bundles this entire workflow into a single object. Instead of calling three different functions separately, you now have one "Run" button (the chain) that does everything in order.

In [33]:
chain = prompt | llm | parser


**chain.invoke(...)**

What it does: This is the "Go" button. It starts the chain we built earlier (prompt | llm | parser).

How it works: It takes your input, pushes it through the prompt template, sends it to the model, and parses the answer—all in one step.

In [34]:
question = "My name is Yasiru"

response = chain.invoke({"question": question})

print(response)

Hi Yasiru! It's nice to meet you.



```python
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
```


---



### **1. from langchain_core.prompts import MessagesPlaceholder**

Used for:
Creating a **placeholder inside a prompt template** where you can insert an entire list of chat messages (such as conversation history).
This is essential when building chatbots or agents that need to remember previous messages in a multi-turn conversation.

It acts like a dynamic slot:

```python
MessagesPlaceholder("chat_history")
```

This tells LangChain to fill in all past messages at runtime.

---

### **2. from langchain_core.messages import HumanMessage**

Used for:
Representing a **message from the user**.
This helps the model understand what the human said, especially when building custom chat histories or testing.

Example:

```python
HumanMessage(content="Hi!")
```

---

### **3. from langchain_core.messages import AIMessage**

Used for:
Representing a **message generated by the AI**.
Helps maintain proper conversation history by distinguishing AI responses from human input.

Example:

```python
AIMessage(content="Hello! How can I help?")
```

---

### **4. from langchain_core.messages import SystemMessage**

Used for:
Defining **rules, instructions, or behavior guidelines** for the AI model.
This message is always processed first and controls how the AI should think or respond.

Example:

```python
SystemMessage(content="You are a helpful assistant.")
```

---




In [35]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage



```python
prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content="You are an intelligent chatbot. Answer the following question."),
        MessagesPlaceholder(variable_name="question")
    ]
)
```

---

### **1. SystemMessage(content="You are an intelligent chatbot. Answer the following question.")**

**Used for:**
Setting the **behavior, rules, and role** of the AI model.

* It tells the AI *how* it should act (intelligent chatbot).
* It instructs the model on its purpose (answer the user's question).
* System messages always come first and shape the model’s personality.

---

### **2. MessagesPlaceholder(variable_name="question")**

**Used for:**
Creating a placeholder in the prompt where **dynamic messages** will be inserted at runtime.

* Here, the variable name `"question"` will hold one or more messages.
* This is often used when you want to pass:

  * user input
  * chat history
  * dynamically generated messages

Even though it's called `"question"`, it represents a **list of messages**, not just one text.

Example runtime use:

```python
prompt.format(question=[HumanMessage(content="What is AI?")])
```

---

### **3. ChatPromptTemplate.from_messages(...)**

**Used for:**
Building a **structured prompt template** that LangChain uses to send messages to the LLM.

* It allows mixing system instructions + dynamic messages.
* Ensures the final prompt is formatted correctly for chat models.

---


In [36]:
prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content="You are an intelligent chatbot. Answer the following question."),
        MessagesPlaceholder(variable_name="question")
    ]
)

In [37]:


# Chain the prompt, LLM, and output parser
chain = prompt | llm | parser

In [38]:
question = "My name is Yasiru"

response = chain.invoke({"question": [HumanMessage(content=question)]})

print(response)

Hello Yasiru! It's nice to meet you. How can I help you today?


In [39]:
question = "Who am I"

response = chain.invoke({"question": [HumanMessage(content=question)]})

print(response)

As an AI, I don't have personal information about you, so I can't tell you who you are in terms of your personal identity, name, or background.

However, I can tell you that you are the individual currently interacting with me, asking this question!

If you're asking in a more philosophical sense, "who am I?" is a profound question that humans often ponder throughout their lives. It can relate to:
*   Your unique personality, values, and beliefs.
*   Your experiences, memories, and history.
*   Your relationships with others.
*   Your goals, dreams, and aspirations.
*   The roles you play in life (e.g., a friend, a professional, a family member).

If you'd like to explore these ideas or talk about aspects of identity, I'm here to chat!


In [40]:
# Create a prompt template with a predefined conversation history and a new question placeholder
prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content="You are an intelligent chatbot. Answer the following question."),
        HumanMessage(content="My name is Yasiru"),
        AIMessage(content="Nice to meet you, Yasiru! How can I assist you today?"),
        MessagesPlaceholder(variable_name="question")
    ]
)

# Chain the prompt, LLM, and output parser
chain = prompt | llm | parser

In [41]:
question = "Who am I"

response = chain.invoke({"question": [HumanMessage(content=question)]})

print(response)

Based on what you've told me, your name is Yasiru!

As an AI, I don't have personal knowledge of you beyond our conversation. The question "Who am I?" is a very deep and personal one that people explore throughout their lives.

It often relates to your experiences, values, relationships, passions, goals, and how you see yourself in the world.

Is there something specific you'd like to explore about yourself or your identity that I can help you think through or provide information on?


In [42]:
# Create a prompt template with a dynamic conversation history and a new question placeholder
prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content="You are an intelligent chatbot. Answer the following question."),
        MessagesPlaceholder(variable_name="history"),
        MessagesPlaceholder(variable_name="question")
    ]
)

# Chain the prompt, LLM, and output parser
chain = prompt | llm | parser

In [43]:
# Define the conversation history
history = [
    HumanMessage(content="My name is Yasiru"),
    AIMessage(content="Nice to meet you, Yasiru! How can I assist you today?"),
    HumanMessage(content="what is 2 + 2"),
    AIMessage(content="4")
]

In [44]:
question = "Who am I"

response = chain.invoke({"history": history, "question": [HumanMessage(content=question)]})
# Extend the history with the latest question and response
history.extend([HumanMessage(content=question), AIMessage(content=response)])
print(response)

Based on our conversation, you are **Yasiru**, the person I am currently interacting with!

As an AI, I only know what you tell me or what we've discussed. I don't have any personal information about you beyond that.


In [45]:
[HumanMessage(content='My name is Yasiru'),
 AIMessage(content='Nice to meet you, Yasiru! How can I assist you today?'),
 HumanMessage(content='what is 2 + 2'),
 AIMessage(content='4'),
 HumanMessage(content='Who am I'),
 AIMessage(content='You are Yasiru.')]

[HumanMessage(content='My name is Yasiru', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Nice to meet you, Yasiru! How can I assist you today?', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what is 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Who am I', additional_kwargs={}, response_metadata={}),
 AIMessage(content='You are Yasiru.', additional_kwargs={}, response_metadata={})]

In [46]:
question = "what's my last question?"

response = chain.invoke({"history": history, "question": [HumanMessage(content=question)]})

history.extend([HumanMessage(content=question), AIMessage(content=response)])

print(response)

Your last question was: "Who am I"


In [47]:
# Display the last four interactions in the conversation history
history[-4:]

[HumanMessage(content='Who am I', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Based on our conversation, you are **Yasiru**, the person I am currently interacting with!\n\nAs an AI, I only know what you tell me or what we've discussed. I don't have any personal information about you beyond that.", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="what's my last question?", additional_kwargs={}, response_metadata={}),
 AIMessage(content='Your last question was: "Who am I"', additional_kwargs={}, response_metadata={})]

In [48]:
question = "what is the gemini ?"

response = chain.invoke({"history": history, "question": [HumanMessage(content=question)]})

history.extend([HumanMessage(content=question), AIMessage(content=response)])

print(response)

"Gemini" can refer to a few different things, so it depends on the context! Here are the most common meanings:

1.  **Google Gemini (AI Model):**
    *   This is very likely what you're asking about, especially since you're talking to an AI!
    *   **What it is:** Google Gemini is a family of multimodal large language models developed by Google AI (specifically Google DeepMind).
    *   **Capabilities:** It's designed to understand and generate not just text, but also code, images, audio, and video. It's built to be highly capable, flexible, and efficient across different types of information.
    *   **My connection:** I am a large language model, trained by Google, and my capabilities are based on the Gemini architecture.

2.  **Gemini (Astrology):**
    *   **What it is:** One of the twelve zodiac signs.
    *   **Dates:** Typically from May 21 to June 20 (dates can vary slightly by year).
    *   **Symbol:** The Twins (representing duality, communication, and intellectual curiosit

In [49]:
question = "how are you "

response = chain.invoke({"history": history, "question": [HumanMessage(content=question)]})

history.extend([HumanMessage(content=question), AIMessage(content=response)])

print(response)

As an AI, I don't have feelings, emotions, or a physical body, so I don't experience "being" in the same way humans do.

However, if you're asking about my operational status, I am functioning perfectly and ready to help you! Thanks for asking.
